In [1]:
! pip install groq pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.6 MB/s eta 0:00:00


In [4]:
"""
classify_response_type_groq.py
-------------------------------
Classifies journal responses to REDIRECTIVE prompts as INTENTION or REFLECTION
using the Groq API (llama-3.3-70b-versatile).

Input  : prompts_classified_task4.csv  (already has prompt_type column)
Output : prompts_with_response_type.csv  — all rows + response_type column
         prompts_intention_only.csv      — REDIRECTIVE + INTENTION rows only

Install:
    pip install groq pandas

Run:
    export GROQ_API_KEY="your_key_here"
    python classify_response_type_groq.py
"""

import json
import os
import time
from pathlib import Path

import pandas as pd
from groq import Groq

# ── Config ────────────────────────────────────────────────────────────────────
INPUT_CSV       = "prompts_classified_task4.csv"
OUT_FULL        = "prompts_with_response_type_task5_a2_3day.csv"   # all rows + response_type
# OUT_INTENTION   = "prompts_intention_only_7day.csv"        # REDIRECTIVE + INTENTION only
CACHE_FILE      = "response_type_cache_1day.json"          # re-run safe cache

MODEL           = "llama-3.3-70b-versatile"
BATCH_SAVE      = 50     # write cache every N API calls
RETRY_LIMIT     = 3
RETRY_DELAY     = 5      # seconds between retries

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are classifying journal responses to AI-generated journaling prompts.

Read the prompt the user received and their journal response, then classify the response into one of three types.

────────────────────────────────────────────────
CATEGORY DEFINITIONS AND EXAMPLES
────────────────────────────────────────────────

INTENTION — The user expresses a forward-looking commitment or plan to change behavior.
The key signal is future-oriented language paired with a specific action.

  Positive examples:
- "It keeps my blood flowing, which is good for my overall health. Though I think I need to work out a little more."
- "I really enjoyed doing bandiegrams today and talking to a larger variety of people in the band itself instead of
 just one or two. Hanging out with my ekt siblings was a lot of fun and I hope we do more things like that in the future,
 I'm very happy I joined the house. I think it's also been good for me to spend more time at the library and do my studying there instead of at home."
- "At this point, I'm trying to shift my habits a little bit, to have a more balanced and healthy life.
The need to change and stop feeling tired all the time is what motivated me to make these changes.
I'm trying to allow my body and mind to rest way before exhaustion."
- "Trying to establish a night routine and sleeping schedule has been really good for me. This past week
I've been trying to go to bed before 22h30, so I can wake up early not feeling tired,
and then have time to exercise or have a longer breakfast. This definitely added some calm to my routine."

  Edge cases that still count as INTENTION:
  • Tentative but future-oriented: "I guess I should try walking more next week" → INTENTION (weak)
  • Goal stated without specific plan: "I want to reduce my screen time this week" → INTENTION (moderate)

  Does NOT count as INTENTION:
  • "I know I should exercise more" (awareness without commitment)
  • "Maybe someday I'll try meditating" (hypothetical, not a real plan)

────────────────────────────────────────────────

REFLECTION — The user reflects on, acknowledges, or describes their current or past behavior
without committing to change. They observe, explain, or accept but do not plan.

  Positive examples:
- "I have been working out before, but I had to take a break because of a foot
injury. It is nice to get back into it, but I feel disheartened because
 I've gotten so much weaker and it feels difficult to get back on the same level."
- "Not really. I have been having great sleep actually. Perhaps I need to stay
  up a little more so that I get going with my work. Otherwise, everything is great."
-  "I think that walking to more places throughout the day makes me feel generally
 happier than when I stay inside."

- "I haven't done any running at all it makes my chest hurt."

  Edge cases that still count as REFLECTION:
  • Insight without follow-through: "I realize I should sleep more, but it's hard" → REFLECTION
  • Explaining why something happened: "My phone was quiet because I forgot it at home" → REFLECTION
  • Agreeing with the prompt without committing: "That's true, I have been less social lately" → REFLECTION

────────────────────────────────────────────────

NOT_APPLICABLE — The response cannot be meaningfully classified as either INTENTION or REFLECTION.
Use this when the response is:
  • Completely off-topic or ignores the prompt entirely
  • Too short or vague to assess (e.g. "yes", "I don't know", "okay")
  • A factual description with no behavioral or emotional language
  • A response to a REFLECTIVE prompt that is purely observational with no personal stance

  Examples:
  • Prompt asks about fitness, user writes about their cat → NOT_APPLICABLE
  • "I'm fine." (no behavioral content) → NOT_APPLICABLE
  • "Today was okay." → NOT_APPLICABLE

────────────────────────────────────────────────

OUTPUT FORMAT — follow this exactly:
REASONING: [2-3 sentences explaining which signals in the response led to your decision. Quote key words or phrases from the response that drove the classification.]
LABEL: [INTENTION or REFLECTION or NOT_APPLICABLE]

Do not add anything after the label line."""


# ── Groq client ───────────────────────────────────────────────────────────────

from google.colab import userdata
from groq import Groq

client = Groq(api_key=userdata.get("GROQ_API_KEY"))

def classify_response(prompt_text: str, response_text: str) -> str:
    """Call Groq API with retry. Returns INTENTION or REFLECTION."""
    user_msg = (
        f'Prompt the user received: "{prompt_text}"\n'
        f'Journal response: "{response_text}"\n'
        f'Respond with only one word: INTENTION or REFLECTION'
    )
    for attempt in range(1, RETRY_LIMIT + 1):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg},
                ],
                max_tokens=20,
                temperature=0.0,
            )
            label = resp.choices[0].message.content.strip().upper()
            if "INTENTION" in label:
                return "INTENTION"
            return "REFLECTION"
        except Exception as e:
            print(f"    Attempt {attempt}/{RETRY_LIMIT} failed: {e}")
            if attempt < RETRY_LIMIT:
                time.sleep(RETRY_DELAY)
    return "REFLECTION"   # safe fallback


def cache_key(prompt_text: str, response_text: str) -> str:
    """Composite key so same response to a different prompt is classified independently."""
    return f"{prompt_text}|||{response_text}"


# ── Load input ────────────────────────────────────────────────────────────────
print(f"Loading: {INPUT_CSV}")
df = pd.read_csv(INPUT_CSV)
print(f"  Shape         : {df.shape}")
print(f"  Columns       : {df.columns.tolist()}")
print(f"  prompt_type   : {df['prompt_type'].value_counts().to_dict()}")

# ── Partition ─────────────────────────────────────────────────────────────────
# Only REDIRECTIVE rows with a non-null response get classified
df_redir   = df[df['prompt_type'] == 'REDIRECTIVE'].copy()
df_other   = df[df['prompt_type'] != 'REDIRECTIVE'].copy()  # REFLECTIVE + null

has_resp   = df_redir['journal_response_text'].notna()
df_to_clf  = df_redir[has_resp].copy()
df_no_resp = df_redir[~has_resp].copy()

print(f"\n  REDIRECTIVE rows            : {len(df_redir)}")
print(f"  → with journal response     : {len(df_to_clf)}")
print(f"  → without journal response  : {len(df_no_resp)}")
print(f"  REFLECTIVE / other rows     : {len(df_other)}")

# ── Load cache ────────────────────────────────────────────────────────────────
cache_path = Path(CACHE_FILE)
cache: dict = json.loads(cache_path.read_text()) if cache_path.exists() else {}

rows_to_run = [
    row for _, row in df_to_clf.iterrows()
    if cache_key(row['prompt_text'], row['journal_response_text']) not in cache
]
print(f"\n  Already cached  : {len(df_to_clf) - len(rows_to_run)}")
print(f"  To classify     : {len(rows_to_run)}")

# ── Classify ──────────────────────────────────────────────────────────────────
print("\nClassifying via Groq...")
total = len(rows_to_run)

for i, row in enumerate(rows_to_run, start=1):
    key = cache_key(row['prompt_text'], row['journal_response_text'])
    cache[key] = classify_response(row['prompt_text'], row['journal_response_text'])

    if i % BATCH_SAVE == 0 or i == total:
        cache_path.write_text(json.dumps(cache, indent=2))
        print(f"  [{i:>4}/{total}] cache saved...")

print(f"\nDone. Cache size: {len(cache)}")

# ── Map labels back ───────────────────────────────────────────────────────────
df_to_clf['response_type']  = df_to_clf.apply(
    lambda r: cache.get(cache_key(r['prompt_text'], r['journal_response_text'])), axis=1
)
df_no_resp['response_type'] = None   # no response → unclassifiable
df_other['response_type']   = None   # non-REDIRECTIVE → not applicable

# Reconstruct full dataframe in original row order
df_out = pd.concat([df_to_clf, df_no_resp, df_other]).sort_index()

# ── Summary ───────────────────────────────────────────────────────────────────
rt = df_out['response_type'].value_counts()
print("\n── Response type results ───────────────────────────────────────")
print(f"  INTENTION          : {rt.get('INTENTION',  0)}")
print(f"  REFLECTION         : {rt.get('REFLECTION', 0)}")
print(f"  Null (not applicable / no response) : {df_out['response_type'].isna().sum()}")

print("\nBreakdown by behavioral domain (REDIRECTIVE rows only):")
domain_breakdown = (
    df_out[df_out['response_type'].notna()]
    .groupby('behavioral_domain_category')['response_type']
    .value_counts()
    .unstack(fill_value=0)
)
print(domain_breakdown)

# ── Export ────────────────────────────────────────────────────────────────────
# Full table — all original columns + response_type
df_out.to_csv(OUT_FULL, index=False)
print(f"\nSaved full table     → {OUT_FULL}  ({len(df_out)} rows, {len(df_out.columns)} columns)")

# INTENTION-only
# df_intention = df_out[df_out['response_type'] == 'INTENTION']
# df_intention.to_csv(OUT_INTENTION, index=False)
# print(f"Saved INTENTION only → {OUT_INTENTION}  ({len(df_intention)} rows)")

Loading: prompts_classified_task4.csv
  Shape         : (648, 11)
  Columns       : ['participant_id', 'week', 'date', 'prompt_text', 'journal_response_text', 'behavioral_domain_category', 'signal_before', 'signal_after', 'signal_change', 'behavioral_improvement', 'prompt_type']
  prompt_type   : {'REFLECTIVE': 536, 'REDIRECTIVE': 112}

  REDIRECTIVE rows            : 112
  → with journal response     : 102
  → without journal response  : 10
  REFLECTIVE / other rows     : 536

  Already cached  : 102
  To classify     : 0

Classifying via Groq...

Done. Cache size: 104

── Response type results ───────────────────────────────────────
  INTENTION          : 54
  REFLECTION         : 48
  Null (not applicable / no response) : 546

Breakdown by behavioral domain (REDIRECTIVE rows only):
response_type               INTENTION  REFLECTION
behavioral_domain_category                       
digital_habits                     14          13
general                             3           0
phys